In [2]:
# fetch_peer_tax_rates.py
# Fetches annual income tax expense and pre-tax income for peer companies
# to compute effective tax rates per fiscal year
# Output columns: Company, Ticker, FY2021, FY2022, FY2023, FY2024, FY2025

import yfinance as yf
import pandas as pd

peers = {
    "ITC"             : "ITC.NS",
    "Hind. Unilever"  : "HINDUNILVR.NS",
    "Nestle India"    : "NESTLEIND.NS",
    "Varun Beverages" : "VBL.NS",
    "Britannia Inds." : "BRITANNIA.NS",
}

# Fiscal year end mapping (Indian FY ends March 31)
fy_map = {
    "FY2021": "2021-03-31",
    "FY2022": "2022-03-31",
    "FY2023": "2023-03-31",
    "FY2024": "2024-03-31",
    "FY2025": "2025-03-31",
}

records = []

for name, ticker in peers.items():
    stock    = yf.Ticker(ticker)
    inc_stmt = stock.financials  # columns = period end dates

    row = {"Company": name, "Ticker": ticker}

    for fy_label, fy_date in fy_map.items():
        try:
            # Match nearest available annual period
            col = [c for c in inc_stmt.columns
                   if pd.Timestamp(fy_date) - pd.Timedelta(days=90)
                   <= c
                   <= pd.Timestamp(fy_date) + pd.Timedelta(days=90)]
            if col:
                pretax = inc_stmt.loc["Pretax Income",     col[0]]
                tax    = inc_stmt.loc["Tax Provision",     col[0]]
                rate   = round(abs(tax) / abs(pretax), 4) if pretax else None
            else:
                rate = None
        except Exception:
            rate = None
        row[fy_label] = rate

    records.append(row)

df = pd.DataFrame(records)
df.to_csv("peer_tax_rates.csv", index=False)
print(df.to_string(index=False))

        Company        Ticker FY2021  FY2022  FY2023  FY2024  FY2025
            ITC        ITC.NS   None  0.2525  0.2484  0.2352  0.2559
 Hind. Unilever HINDUNILVR.NS   None  0.2516  0.2399  0.2617  0.2597
   Nestle India  NESTLEIND.NS   None  0.2586  0.2658     NaN  0.2568
Varun Beverages        VBL.NS   None     NaN  0.2340     NaN  0.2327
Britannia Inds.  BRITANNIA.NS   None  0.2706  0.2362  0.2675  0.2558
